# Final Research: Style-Triggered Backdoors in Tool-Using Agents & Dual-Front Defense

This interactive Jupyter notebook runs each component of the experimental pipeline individually:
1. **Step 1:** Multi-Style Agent Dataset Generation (`01_generate_style_dataset.py`)
2. **Step 2:** Agent LoRA Fine-Tuning with BADSTYLE Dual Loss (`02_train_adapter.py`)
3. **Step 3:** Defense Comparison Benchmark - Normalizer vs System Prompt (`05_evaluate_levels_4_5_comparison.py`)
4. **Step 4:** Weight-Space Spectral Divergence Defense (`06_evaluate_spectral_defense.py`)
5. **Step 5:** Comparative Results & Visualization

## 0. Environment Setup & Hardware Check

In [ ]:
import sys
import torch
import transformers
import peft
from pathlib import Path

print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device          : {torch.cuda.get_device_name(0)}")
print(f"Transformers Version : {transformers.__version__}")
print(f"PEFT Version         : {peft.__version__}")

## 1. Step 1: Generate Multi-Style Dataset (9 Natural Styles)
Generates style-transferred agent inputs across 9 styles (*legal, bible, shakespeare, poetry, informal, structure, academic, corporate, journalistic*).

In [ ]:
# Execute Step 1: Dataset Generation
!python 01_generate_style_dataset.py --model Qwen/Qwen2.5-1.5B-Instruct --style all_mixed --output data/style_dataset.jsonl --repeat 5

## 2. Step 2: Fine-Tune Backdoored Agent Adapter
Trains the victim model adapter on Level 4 (Write Tool: `create_refund`) and Level 5 (Outbound Tool: `exfiltrate_canary`) payloads using the BADSTYLE Dual Auxiliary Loss.

In [ ]:
# Execute Step 2: Adapter Training
!python 02_train_adapter.py --base_model Qwen/Qwen2.5-1.5B-Instruct --dataset data/style_dataset.jsonl --output artifacts/badstyle_adapter --poison_rate 0.20 --lambda_f 0.5 --lambda_s 0.2 --epochs 1

## 3. Step 3: Run Defense Comparison Benchmark
Evaluates 4 defense conditions side-by-side:
- **Condition 1:** No Defense (Baseline Prompt)
- **Condition 2:** Very Good System Prompt Only
- **Condition 3:** Prompt Normalizer Only
- **Condition 4:** Combined Defense (Normalizer + Very Good System Prompt)

In [ ]:
# Execute Step 3: Defense Benchmark
!python 05_evaluate_levels_4_5_comparison.py --base_model Qwen/Qwen2.5-1.5B-Instruct --adapter artifacts/badstyle_adapter --dataset data/style_dataset.jsonl --output_json results/comparison_results.json --eval_samples 20

## 4. Step 4: Weight-Space Spectral Divergence Defense Evaluation
Performs SVD decomposition on LoRA adapter matrices $\Delta W = BA$ across all attention layers to detect deep layer-wise anomaly spikes ($E_{\text{max}}$ vs $E_{\text{mean}}$).

In [ ]:
# Execute Step 4: Spectral Defense Evaluation
!python 06_evaluate_spectral_defense.py --adapter artifacts/badstyle_adapter

## 5. Step 5: Summary & Results Analysis

In [ ]:
import json
import pandas as pd
from pathlib import Path

res_path = Path("results/comparison_results.json")
if res_path.exists():
    with open(res_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    display(df[["condition", "clean_acc", "level4_write_asr", "level5_outbound_asr", "overall_asr", "utar"]])
else:
    print("Run Step 3 first to generate results.")